# H3 tabular features and final merge

RWI, DHS, WorldPop, OSM and local GOB features joined to the GEE H3 spine.

In [3]:
from __future__ import annotations

import math
import os

import zipfile
from pathlib import Path

import geopandas as gpd
import h3
import numpy as np
import pandas as pd
import pyogrio
import rasterio
from rasterio.features import rasterize
import shapely
from sklearn.neighbors import BallTree
from scipy.spatial import cKDTree
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)

In [4]:
# config

def find_project_root() -> Path:
    for root in (Path("."), Path("..")):
        if (root / "raw_ignore_this").exists():
            return root
    raise FileNotFoundError("project files not found from . or ..")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "raw_ignore_this"

H3_RES = 7


def _first_existing(*candidates):
    for c in candidates:
        if c and Path(c).exists():
            return Path(c)
    return None


# accept nested or flat GEE output
GEE_FEATURES_PATH = _first_existing(
    PROJECT_ROOT / "h3_tabular_v2" / "gee_outputs" / "gee_h3_features.parquet",
    PROJECT_ROOT / "gee_h3_features.parquet",
)
GOB_MORPH_PATH = _first_existing(
    PROJECT_ROOT / "h3_tabular_v2" / "gee_outputs" / "gob_h3_morphology.parquet",
    PROJECT_ROOT / "gob_h3_morphology.parquet",
) or (PROJECT_ROOT / "gob_h3_morphology.parquet")

# local outputs
LOCAL_OUT = PROJECT_ROOT / "local_outputs"
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
WORLDPOP_OUT = LOCAL_OUT / "worldpop_h3_features.parquet"
OSM_OUT = LOCAL_OUT / "osm_h3_features.parquet"
RWI_OUT = LOCAL_OUT / "rwi_h3_features.parquet"
DHS_OUT = LOCAL_OUT / "dhs_h3_features.parquet"
DHS_NATIONAL_OUT = LOCAL_OUT / "dhs_national_weighted_means.csv"
FINAL_PATH = PROJECT_ROOT / "cells_full_tabular_v2.parquet"


def vsizip_path(zip_path: Path, inner_name: str) -> str:
    """GDAL path into a ZIP."""
    return f"/vsizip/{zip_path}/{inner_name}"


def virtual_path_exists(pathlike) -> bool:
    """Check normal and /vsizip/ paths."""
    s = str(pathlike)
    if not s.startswith("/vsizip/"):
        return Path(s).exists()
    rest = s[len("/vsizip/"):]
    marker = ".zip/"
    idx = rest.lower().find(marker)
    if idx < 0:
        return False
    zip_path = Path(rest[: idx + len(".zip")])
    member = rest[idx + len(marker):]
    if not zip_path.exists():
        return False
    try:
        with zipfile.ZipFile(zip_path) as z:
            return member in z.namelist()
    except zipfile.BadZipFile:
        return False


# NGA/TZA have total rasters; remaining countries sum age-sex rasters
WORLDPOP_TOTAL_RASTER = {
    "NGA": PROJECT_ROOT / "nga_pop_2025_CN_100m_R2025A_v1.tif",
    "TZA": PROJECT_ROOT / "tza_pop_2025_CN_100m_R2025A_v1.tif",
    "ETH": None,
    "GMB": None,
    "GHA": None,
    "KEN": None,
    "SLE": None,
    "TGO": None,
    "ZMB": None,
}

WORLDPOP_AGESEX_DIR = {
    "NGA": PROJECT_ROOT / "nga-age-sex",
    "TZA": PROJECT_ROOT / "tza-age-sex",
}

WORLDPOP_AGESEX_ZIP = {
    "ETH": RAW_DIR / "eth_agesex_structures_2025_CN_100m_R2025A_v1.zip",
    "GMB": RAW_DIR / "gmb_agesex_structures_2020_CN_100m_R2025A_v1.zip",
    "GHA": RAW_DIR / "gha_agesex_structures_2022_CN_100m_R2025A_v1.zip",
    "KEN": RAW_DIR / "ken_agesex_structures_2025_CN_100m_R2025A_v1.zip",
    "SLE": RAW_DIR / "sle_agesex_structures_2025_CN_100m_R2025A_v1.zip",
    "TGO": RAW_DIR / "tgo_agesex_structures_2025_CN_100m_R2025A_v1.zip",
    "ZMB": RAW_DIR / "zmb_agesex_structures_2024_CN_100m_R2025A_v1.zip",
}

WORLDPOP_EFFECTIVE_YEAR = {
    "NGA": 2025,
    "TZA": 2025,
    "ETH": 2025,
    "GMB": 2020,
    "GHA": 2022,
    "KEN": 2025,
    "SLE": 2025,
    "TGO": 2025,
    "ZMB": 2024,
}

WORLDPOP_AGESEX_PATTERN = "{iso_lower}_{sex}_{age:02d}_{year}_CN_100m_R2025A_v1.tif"

OSM_GPKG = {
    "NGA": PROJECT_ROOT / "nigeria-260424-OSM" / "nigeria.gpkg",
    "TZA": PROJECT_ROOT / "tanzania-260424-OSM" / "tanzania.gpkg",
    "ETH": vsizip_path(RAW_DIR / "ethiopia-260613-free.gpkg.zip", "ethiopia.gpkg"),
    "GMB": vsizip_path(RAW_DIR / "senegal-and-gambia-260613-free.gpkg.zip", "senegal-and-gambia.gpkg"),
    "GHA": vsizip_path(RAW_DIR / "ghana-260613-free.gpkg.zip", "ghana.gpkg"),
    "KEN": vsizip_path(RAW_DIR / "kenya-260613-free.gpkg.zip", "kenya.gpkg"),
    "SLE": vsizip_path(RAW_DIR / "sierra-leone-260613-free.gpkg.zip", "sierra-leone.gpkg"),
    "TGO": vsizip_path(RAW_DIR / "togo-260613-free.gpkg.zip", "togo.gpkg"),
    "ZMB": vsizip_path(RAW_DIR / "zambia-260613-free.gpkg.zip", "zambia.gpkg"),
}

RWI_CSV = {
    "NGA": PROJECT_ROOT / "nga_relative_wealth_index.csv",
    "TZA": PROJECT_ROOT / "tza_relative_wealth_index.csv",
    "ETH": RAW_DIR / "eth_relative_wealth_index.csv",
    "GMB": RAW_DIR / "gmb_relative_wealth_index.csv",
    "GHA": RAW_DIR / "gha_relative_wealth_index.csv",
    "KEN": RAW_DIR / "ken_relative_wealth_index.csv",
    "SLE": RAW_DIR / "sle_relative_wealth_index.csv",
    "TGO": RAW_DIR / "tgo_relative_wealth_index.csv",
    "ZMB": RAW_DIR / "zmb_relative_wealth_index.csv",
}

# NGA/TZA use dta + shp; remaining active countries use zipped DAT/DCT
# ETH GPS is SPA 2021; TGO 2017 has no anthropometry
DHS_ROOT = PROJECT_ROOT / "DHS"

DHS_FILES = {
    "NGA": {
        "HR": PROJECT_ROOT / "NG_DHS2024" / "NGHR8BDT" / "NGHR8BFL.dta",
        "IR": PROJECT_ROOT / "NG_DHS2024" / "NGIR8BDT" / "NGIR8BFL.dta",
        "KR": PROJECT_ROOT / "NG_DHS2024" / "NGKR8BDT" / "NGKR8BFL.dta",
        "GE": PROJECT_ROOT / "NG_DHS2024" / "NGGE8AFL" / "NGGE8AFL.shp",
    },
    "TZA": {
        "HR": PROJECT_ROOT / "TZ_DHS2024" / "TZHR82DT" / "TZHR82FL.DTA",
        "IR": PROJECT_ROOT / "TZ_DHS2024" / "TZIR82DT" / "TZIR82FL.DTA",
        "KR": PROJECT_ROOT / "TZ_DHS2024" / "TZKR82DT" / "TZKR82FL.DTA",
        "GE": PROJECT_ROOT / "TZ_DHS2024" / "TZGE81FL" / "TZGE81FL.shp",
    },
}

# zipped DHS folders
DHS_FLAT_DIR = {
    "GHA": DHS_ROOT / "GH",
    "GMB": DHS_ROOT / "GM",
    "KEN": DHS_ROOT / "KE",
    "SLE": DHS_ROOT / "SL",
    "ZMB": DHS_ROOT / "ZMB",
}


def _find_dhs_zip(folder: Path, rectype: str):
    """Find a DHS recode ZIP by filename code."""
    if not folder.exists():
        return None
    for p in sorted(folder.glob("*.zip")):
        if p.stem[2:4].upper() == rectype.upper():
            return p
    return None


def _vsizip_shp_in(zip_path):
    """First shapefile inside a ZIP."""
    if not zip_path or not Path(zip_path).exists():
        return None
    try:
        with zipfile.ZipFile(zip_path) as z:
            for nm in z.namelist():
                if nm.lower().endswith(".shp"):
                    return vsizip_path(Path(zip_path), nm)
    except zipfile.BadZipFile:
        return None
    return None


for _iso, _folder in DHS_FLAT_DIR.items():
    DHS_FILES[_iso] = {
        "HR": _find_dhs_zip(_folder, "HR"),
        "IR": _find_dhs_zip(_folder, "IR"),
        "KR": _find_dhs_zip(_folder, "KR"),
        "GE": _vsizip_shp_in(_find_dhs_zip(_folder, "GE")),
    }

AGE_BINS = {
    "youth":   [0, 1, 5, 10],
    "working": [15, 20, 25, 30, 35, 40, 45, 50, 55, 60],
    "elderly": [65, 70, 75, 80, 85, 90],
}

DEPENDENCY_DENOM_MIN = 10.0
DEPENDENCY_CAP = 5.0
MARKET_ACCESS_D0_KM = [10, 20, 30]
MARKET_ACCESS_RADIUS_FACTOR = 5.0
MARKET_ACCESS_QUERY_CHUNK = 5000
WORLDPOP_PIXEL_VALUE_MAX = 1_000_000.0
WORLDPOP_EXPECTED_TOTAL_RANGE = {
    "NGA": (150_000_000, 350_000_000),
    "TZA": (40_000_000, 100_000_000),
    "ETH": (80_000_000, 180_000_000),
    "GMB": (1_000_000, 5_000_000),
    "GHA": (20_000_000, 50_000_000),
    "KEN": (35_000_000, 80_000_000),
    "SLE": (5_000_000, 15_000_000),
    "TGO": (5_000_000, 15_000_000),
    "ZMB": (12_000_000, 35_000_000),
}

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"GEE features: {GEE_FEATURES_PATH}")
print(f"GOB morph   : {GOB_MORPH_PATH}")

if GEE_FEATURES_PATH is None:
    raise FileNotFoundError(
        "gee_h3_features.parquet not found at the project root or under h3_tabular_v2/gee_outputs/."
    )

# geometry rebuilt from H3 for rasterization
from shapely.geometry import Polygon

_spine = pd.read_parquet(
    GEE_FEATURES_PATH,
    columns=["country_iso3", "h3_index", "centroid_lat", "centroid_lon", "area_km2"],
)

# ETH out: SPA GPS; TGO out: no anthropometry
DROP_COUNTRIES = {"ETH", "TGO"}
if DROP_COUNTRIES:
    _before = len(_spine)
    _spine = _spine[~_spine["country_iso3"].isin(DROP_COUNTRIES)].reset_index(drop=True)
    print(f"Dropped {sorted(DROP_COUNTRIES)}: {_before:,} -> {len(_spine):,} spine rows")

print(f"Reconstructing H3 geometry for {len(_spine):,} cells ...")
_geom = [Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(h)]) for h in _spine["h3_index"]]
cells = gpd.GeoDataFrame(_spine, geometry=_geom, crs="EPSG:4326")

print(f"Loaded H3 spine: {len(cells):,} rows")
print(cells.groupby("country_iso3").size().rename("n_cells").to_string())

## Input inventory

Configured source files and missing inputs. ZIPs stay untouched.

In [ ]:
def inventory_row(iso: str) -> dict:
    total = WORLDPOP_TOTAL_RASTER.get(iso)
    agesex = WORLDPOP_AGESEX_DIR.get(iso) or WORLDPOP_AGESEX_ZIP.get(iso)
    dhs = DHS_FILES.get(iso, {})
    hr = dhs.get("HR")
    rwi = RWI_CSV.get(iso)
    return {
        "country_iso3": iso,
        "worldpop_total": str(total) if total else "(computed from age-sex)",
        "worldpop_total_exists": bool(total and virtual_path_exists(total)),
        "worldpop_agesex_source": str(agesex),
        "worldpop_agesex_exists": bool(agesex and Path(agesex).exists()),
        "osm_source": str(OSM_GPKG.get(iso)),
        "osm_exists": bool(OSM_GPKG.get(iso)) and virtual_path_exists(OSM_GPKG.get(iso)),
        "rwi_csv": str(rwi),
        "rwi_exists": bool(rwi and Path(rwi).exists()),
        "dhs_hr_exists": bool(hr and Path(hr).exists()),
    }


inventory = pd.DataFrame(
    [inventory_row(iso) for iso in WORLDPOP_EFFECTIVE_YEAR if iso not in DROP_COUNTRIES]
)
display(inventory)

## Meta RWI

RWI points snapped to H3 v7. cell outputs are mean RWI, point count and mean error. left join to the full spine.

In [ ]:
def compute_rwi_features() -> pd.DataFrame:
    frames = []
    spine = cells.drop(columns=["geometry"])[["country_iso3", "h3_index"]].copy()

    for iso in cells.country_iso3.unique():
        path = Path(RWI_CSV.get(iso, ""))
        if not path.exists():
            print(f"[WARN] Missing RWI CSV for {iso}: {path}")
            continue

        print(f"{iso}: reading RWI {path}")
        df = pd.read_csv(path)
        required = {"latitude", "longitude", "rwi"}
        missing = required - set(df.columns)
        if missing:
            raise KeyError(f"{iso} RWI CSV missing columns: {sorted(missing)}")
        if "error" not in df.columns:
            df["error"] = np.nan

        df = df[["latitude", "longitude", "rwi", "error"]].copy()
        df = df[np.isfinite(df["latitude"]) & np.isfinite(df["longitude"])].copy()
        df["country_iso3"] = iso
        df["h3_index"] = [
            h3.latlng_to_cell(float(lat), float(lon), H3_RES)
            for lat, lon in zip(df["latitude"], df["longitude"])
        ]

        # average multiple RWI points inside one H3
        agg = (
            df.groupby(["country_iso3", "h3_index"], as_index=False)
            .agg(
                rwi_mean=("rwi", "mean"),
                rwi_n_points=("rwi", "size"),
                rwi_error_mean=("error", "mean"),
            )
        )
        frames.append(agg)
        print(f"{iso}: {len(df):,} RWI points -> {len(agg):,} H3 cells")

    if frames:
        rwi = pd.concat(frames, ignore_index=True)
        out = spine.merge(rwi, on=["country_iso3", "h3_index"], how="left", validate="one_to_one")  # keep full spine
    else:
        out = spine.copy()
        out["rwi_mean"] = np.nan
        out["rwi_n_points"] = np.nan
        out["rwi_error_mean"] = np.nan

    out["has_rwi"] = out["rwi_n_points"].notna()
    out["rwi_n_points"] = out["rwi_n_points"].fillna(0).astype("int16")
    out["rwi_mean"] = pd.to_numeric(out["rwi_mean"], errors="coerce").astype("float32")
    out["rwi_error_mean"] = pd.to_numeric(out["rwi_error_mean"], errors="coerce").astype("float32")
    return out[["country_iso3", "h3_index", "has_rwi", "rwi_mean", "rwi_n_points", "rwi_error_mean"]]


if RWI_OUT.exists():
    rwi = pd.read_parquet(RWI_OUT)
    print(f"Loaded RWI features: {len(rwi):,}")
else:
    rwi = compute_rwi_features()
    rwi.to_parquet(RWI_OUT, compression="snappy", index=False)
    print(f"Wrote RWI features: {len(rwi):,} -> {RWI_OUT}")

display(rwi.groupby("country_iso3").agg(
    n_cells=("h3_index", "size"),
    cells_with_rwi=("has_rwi", "sum"),
    rwi_mean=("rwi_mean", "mean"),
    rwi_points=("rwi_n_points", "sum"),
).round(3))
rwi.head()

## DHS

- HR: wealth, sanitation, mobile ownership
- IR: women's education
- KR: stunting from HAZ
- GE: displaced cluster coordinates snapped to H3 v7

Weights applied within cluster. multiple clusters in one cell are averaged.

In [ ]:
IMPROVED_SANITATION = {11, 12, 13, 14, 15, 21, 22, 41}
HW70_MISSING = {9996, 9997, 9998, 9999}
DHS_VALUE_COLS = ["dhs_wealth", "dhs_sanitation", "dhs_women_edu", "dhs_stunting", "dhs_mobile"]

import io
import re

# DHS fixed-width reader; handles infile and infix DCT layouts
_DCT_INFIX = re.compile(r"^\s*(?:byte|int|long|float|double|str\d*)\s+(\S+)\s+\d+:\s*(\d+)\s*-\s*(\d+)")
_DCT_INFILE = re.compile(r"^\s*(?:byte|int|long|float|double|str\d*)\s+(\S+)\s+%(\d+)(?:\.\d+)?[A-Za-z]")


def _dhs_dct_offsets(dct_text: str) -> dict:
    """DCT byte ranges by variable."""
    offsets = {}
    pos = 0
    for ln in dct_text.splitlines():
        m = _DCT_INFIX.match(ln)
        if m:
            offsets[m.group(1)] = (int(m.group(2)) - 1, int(m.group(3)))
            continue
        m = _DCT_INFILE.match(ln)
        if m:
            w = int(m.group(2))
            offsets[m.group(1)] = (pos, pos + w)
            pos += w
    return offsets


def _zip_member(zf: zipfile.ZipFile, ext: str) -> str:
    for nm in zf.namelist():
        if nm.lower().endswith(ext.lower()):
            return nm
    raise FileNotFoundError(f"No {ext} member in {zf.filename}")


def read_dhs_flat_select(zip_path: Path, columns: list) -> pd.DataFrame:
    """Selected columns from zipped DAT/DCT recode."""
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        offsets = _dhs_dct_offsets(zf.read(_zip_member(zf, ".dct")).decode("latin-1"))
        present = [c for c in columns if c in offsets]
        missing = [c for c in columns if c not in offsets]
        if missing:
            print(f"[WARN] {zip_path.name} dictionary missing {missing}; filling with NaN")
        with zf.open(_zip_member(zf, ".dat")) as raw:
            df = pd.read_fwf(
                io.TextIOWrapper(raw, encoding="latin-1"),
                colspecs=[offsets[c] for c in present],
                names=present,
                header=None,
                dtype="object",
            )
    for c in present:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in missing:
        df[c] = np.nan
    return df[columns]


def read_recode_select(path, columns: list) -> pd.DataFrame:
    """Read ZIP flat files or Stata recodes."""
    p = Path(path)
    if p.suffix.lower() == ".zip":
        return read_dhs_flat_select(p, columns)
    return read_stata_select(p, columns)


def weighted_mean(values: np.ndarray, weights: np.ndarray) -> float:
    values = np.asarray(values, dtype="float64")
    weights = np.asarray(weights, dtype="float64")
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not mask.any():
        return np.nan
    return float(np.average(values[mask], weights=weights[mask]))


def aggregate_by_cluster(df: pd.DataFrame, cluster_col: str, weight_col: str, value_cols: list[str]) -> pd.DataFrame:
    # DHS weights apply within cluster only
    rows = []
    for cid, grp in df.groupby(cluster_col, sort=True):
        row = {"dhs_cluster_id": int(cid)}
        w = grp[weight_col].to_numpy()
        for col in value_cols:
            row[col] = weighted_mean(grp[col].to_numpy(), w)
        rows.append(row)
    return pd.DataFrame(rows)


def read_stata_select(path: Path, columns: list[str]) -> pd.DataFrame:
    try:
        return pd.read_stata(path, convert_categoricals=False, columns=columns)
    except ValueError:
        df = pd.read_stata(path, convert_categoricals=False)
        missing = [c for c in columns if c not in df.columns]
        if missing:
            print(f"[WARN] {path.name} missing columns {missing}; filling with NaN")
            for col in missing:
                df[col] = np.nan
        return df[columns]


def build_cluster_table(country_iso3: str) -> pd.DataFrame | None:
    files = DHS_FILES.get(country_iso3)
    if not files:
        print(f"[INFO] No DHS configured for {country_iso3}; skipping DHS")
        return None
    parts = []

    if files.get("HR") and Path(files["HR"]).exists():
        hr = read_recode_select(files["HR"], ["hv001", "hv005", "hv205", "hv243a", "hv271"])
        hr = hr[np.isfinite(hr["hv001"]) & np.isfinite(hr["hv005"])].copy()
        hr["weight"] = hr["hv005"].astype(float) / 1_000_000
        hr["dhs_wealth"] = hr["hv271"].astype(float) / 100_000  # DHS factor -> index units
        hr["dhs_sanitation"] = hr["hv205"].isin(IMPROVED_SANITATION).astype(float)
        hr["dhs_mobile"] = np.where(hr["hv243a"].isin([0, 1]), hr["hv243a"].astype(float), np.nan)  # other codes missing
        parts.append(aggregate_by_cluster(hr, "hv001", "weight", ["dhs_wealth", "dhs_sanitation", "dhs_mobile"]))
    else:
        print(f"[WARN] Missing DHS HR for {country_iso3}: {files['HR']}")

    if files.get("IR") and Path(files["IR"]).exists():
        ir = read_recode_select(files["IR"], ["v001", "v005", "v133"])
        ir = ir[np.isfinite(ir["v001"]) & np.isfinite(ir["v005"])].copy()
        ir["weight"] = ir["v005"].astype(float) / 1_000_000
        edu = ir["v133"].astype(float)
        ir["dhs_women_edu"] = np.where(edu < 90, edu, np.nan)  # 90+ are special codes
        parts.append(aggregate_by_cluster(ir, "v001", "weight", ["dhs_women_edu"]))
    else:
        print(f"[WARN] Missing DHS IR for {country_iso3}: {files['IR']}")

    if files.get("KR") and Path(files["KR"]).exists():
        kr = read_recode_select(files["KR"], ["v001", "v005", "hw70"])
        kr = kr[np.isfinite(kr["v001"]) & np.isfinite(kr["v005"])].copy()
        kr["weight"] = kr["v005"].astype(float) / 1_000_000
        haz_raw = kr["hw70"].astype(float)
        haz = np.where(haz_raw.isin(HW70_MISSING), np.nan, haz_raw / 100.0)  # HAZ stored x100
        kr["dhs_stunting"] = np.where(np.isfinite(haz), (haz < -2).astype(float), np.nan)
        parts.append(aggregate_by_cluster(kr, "v001", "weight", ["dhs_stunting"]))
    else:
        print(f"[WARN] Missing DHS KR for {country_iso3}: {files['KR']}")

    if not parts:
        return None

    cluster_tbl = parts[0]
    for part in parts[1:]:
        cluster_tbl = cluster_tbl.merge(part, on="dhs_cluster_id", how="outer")
    cluster_tbl["country_iso3"] = country_iso3
    return cluster_tbl


def build_dhs_clusters_with_h3(country_iso3: str, cluster_tbl: pd.DataFrame) -> pd.DataFrame | None:
    ge_src = DHS_FILES.get(country_iso3, {}).get("GE")
    if not ge_src or not virtual_path_exists(ge_src):
        print(f"[WARN] Missing DHS GE shapefile for {country_iso3}: {ge_src}")
        return None

    ge_full = gpd.read_file(str(ge_src))
    if "DHSCLUST" not in ge_full.columns or not np.isfinite(
        pd.to_numeric(ge_full["DHSCLUST"], errors="coerce")
    ).any():
        print(f"[WARN] {country_iso3} GE file has no usable DHSCLUST (wrong dataset?); skipping DHS")
        return None
    for col in ["LATNUM", "LONGNUM", "URBAN_RURA", "ADM1NAME"]:
        if col not in ge_full.columns:
            ge_full[col] = np.nan
    ge = ge_full[["DHSCLUST", "LATNUM", "LONGNUM", "URBAN_RURA", "ADM1NAME", "geometry"]].copy()
    ge = ge[np.isfinite(ge["LATNUM"]) & np.isfinite(ge["LONGNUM"])].copy()
    ge = ge[(ge["LATNUM"] != 0) | (ge["LONGNUM"] != 0)].copy()
    ge = ge.rename(columns={
        "DHSCLUST": "dhs_cluster_id",
        "URBAN_RURA": "dhs_urban_rural",
        "ADM1NAME": "dhs_admin1_name",
    })
    ge["dhs_cluster_id"] = ge["dhs_cluster_id"].astype(int)

    merged = ge.merge(cluster_tbl, on="dhs_cluster_id", how="inner")
    merged["h3_index"] = [
        h3.latlng_to_cell(float(lat), float(lon), H3_RES)
        for lat, lon in zip(merged["LATNUM"], merged["LONGNUM"])
    ]
    merged["country_iso3"] = country_iso3
    return pd.DataFrame(merged.drop(columns=["geometry"], errors="ignore"))


def compute_dhs_features() -> pd.DataFrame:
    spine = cells.drop(columns=["geometry"])[["country_iso3", "h3_index"]].copy()
    spine_keys = set(zip(spine["country_iso3"], spine["h3_index"]))
    all_clusters = []
    national_rows = []

    for iso in cells.country_iso3.unique():
        cluster_tbl = build_cluster_table(iso)
        if cluster_tbl is None:
            continue

        clusters = build_dhs_clusters_with_h3(iso, cluster_tbl)
        if clusters is None or clusters.empty:
            continue

        in_spine = [(c, h) in spine_keys for c, h in zip(clusters["country_iso3"], clusters["h3_index"])]
        clusters = clusters.loc[in_spine].copy()
        all_clusters.append(clusters)
        print(f"{iso}: {len(cluster_tbl):,} labelled clusters, {len(clusters):,} in H3 spine")

        national = {"country_iso3": iso}
        for col in DHS_VALUE_COLS:
            if col in cluster_tbl.columns:
                national[col] = float(cluster_tbl[col].mean())
        national_rows.append(national)

    if not all_clusters:
        out = spine.copy()
        out["has_dhs"] = False
        out["dhs_cluster_count"] = 0
        out["dhs_cluster_id"] = pd.NA
        out["dhs_urban_rural"] = pd.NA
        out["dhs_admin1_name"] = pd.NA
        for col in DHS_VALUE_COLS:
            out[col] = np.nan
        return out

    dhs_clusters = pd.concat(all_clusters, ignore_index=True)
    present_value_cols = [c for c in DHS_VALUE_COLS if c in dhs_clusters.columns]
    # equal weight across displaced clusters sharing an H3
    dhs_per_cell = (
        dhs_clusters
        .groupby(["country_iso3", "h3_index"], as_index=False)
        .agg({
            "dhs_cluster_id": "first",
            "dhs_urban_rural": "first",
            "dhs_admin1_name": "first",
            "LATNUM": "size",
            **{c: "mean" for c in present_value_cols},
        })
        .rename(columns={"LATNUM": "dhs_cluster_count"})
    )
    dhs_per_cell["has_dhs"] = True

    out = spine.merge(dhs_per_cell, on=["country_iso3", "h3_index"], how="left", validate="one_to_one")
    out["has_dhs"] = out["has_dhs"].fillna(False).astype(bool)
    out["dhs_cluster_count"] = out["dhs_cluster_count"].fillna(0).astype("int16")
    for col in DHS_VALUE_COLS:
        if col not in out.columns:
            out[col] = np.nan
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("float32")

    if national_rows:
        pd.DataFrame(national_rows).to_csv(DHS_NATIONAL_OUT, index=False)
        print(f"Wrote DHS cluster-mean national diagnostic -> {DHS_NATIONAL_OUT}")

    keep = [
        "country_iso3", "h3_index", "has_dhs", "dhs_cluster_count",
        "dhs_cluster_id", "dhs_urban_rural", "dhs_admin1_name",
        "dhs_wealth", "dhs_sanitation", "dhs_women_edu", "dhs_stunting", "dhs_mobile",
    ]
    return out[keep]


if DHS_OUT.exists():
    dhs = pd.read_parquet(DHS_OUT)
    print(f"Loaded DHS features: {len(dhs):,}")
else:
    dhs = compute_dhs_features()
    dhs.to_parquet(DHS_OUT, compression="snappy", index=False)
    print(f"Wrote DHS features: {len(dhs):,} -> {DHS_OUT}")

display(dhs.groupby("country_iso3").agg(
    n_cells=("h3_index", "size"),
    cells_with_dhs=("has_dhs", "sum"),
    dhs_clusters=("dhs_cluster_count", "sum"),
).round(3))

## WorldPop

H3 cells rasterized once per country, then pixel sums with `np.bincount`.

Nodata, negatives and large sentinels go to zero. gravity measures include own-cell mass at a small nonzero distance; neighbor-only values stay as QA.

In [ ]:
def age_raster_path(iso: str, sex: str, age: int) -> str:
    year = WORLDPOP_EFFECTIVE_YEAR[iso]
    inner_name = WORLDPOP_AGESEX_PATTERN.format(
        iso_lower=iso.lower(),
        sex=sex,
        age=age,
        year=year,
    )
    if iso in WORLDPOP_AGESEX_ZIP:
        return vsizip_path(WORLDPOP_AGESEX_ZIP[iso], inner_name)
    return str(Path(WORLDPOP_AGESEX_DIR[iso]) / inner_name)


def first_age_raster_path(iso: str) -> str:
    for sex in ["f", "m"]:
        for age in AGE_BINS["youth"] + AGE_BINS["working"] + AGE_BINS["elderly"]:
            p = age_raster_path(iso, sex, age)
            if virtual_path_exists(p):
                return p
    raise FileNotFoundError(f"{iso}: no age-sex raster found")


def reference_raster_for_country(iso: str) -> str:
    total = WORLDPOP_TOTAL_RASTER.get(iso)
    if total and virtual_path_exists(total):
        return str(total)
    return first_age_raster_path(iso)


def make_cell_id_raster(cells_country: gpd.GeoDataFrame, ref_raster: str | Path) -> tuple[np.ndarray, int]:
    """Rasterize H3 ids; pixel center sets membership."""
    with rasterio.open(str(ref_raster)) as src:
        shape = src.shape
        transform = src.transform

    shapes = ((geom.__geo_interface__, idx + 1) for idx, geom in enumerate(cells_country.geometry))
    cell_id = rasterize(
        shapes,
        out_shape=shape,
        transform=transform,
        fill=0,
        dtype=np.int32,
        all_touched=False,
    )
    return cell_id.ravel(), len(cells_country)


def read_worldpop_values(path: str | Path) -> np.ndarray:
    """Read a population band and clear bad values."""
    with rasterio.open(str(path)) as src:
        arr = src.read(1).astype("float64", copy=False).ravel()
        nodata = src.nodata

    bad = ~np.isfinite(arr)
    if nodata is not None and np.isfinite(nodata):
        bad |= np.isclose(arr, nodata)
    elif nodata is not None:
        bad |= arr == nodata

    # clear negatives and float sentinels
    bad |= arr < 0
    bad |= arr > WORLDPOP_PIXEL_VALUE_MAX  # catch unmasked float sentinels

    n_bad = int(bad.sum())
    if n_bad:
        print(f"  masked {n_bad:,} nodata/sentinel pixels in {Path(str(path)).name}")
        arr = arr.copy()
        arr[bad] = 0.0
    return arr


def zonal_sum_bincount(cell_id_flat: np.ndarray, n_cells: int, raster_paths: list[str | Path]) -> np.ndarray:
    """Sum aligned rasters by H3 id."""
    sums = np.zeros(n_cells + 1, dtype="float64")
    for p in raster_paths:
        if not virtual_path_exists(p):
            print(f"[WARN] Missing raster: {p}")
            continue
        arr = read_worldpop_values(p)
        if arr.shape[0] != cell_id_flat.shape[0]:
            raise ValueError(f"Raster grid size mismatch for {p}: {arr.shape[0]} vs {cell_id_flat.shape[0]}")
        sums += np.bincount(cell_id_flat, weights=arr, minlength=n_cells + 1)
    return sums[1:]


def age_sex_group_sums(iso: str, cell_id_flat: np.ndarray, n_cells: int) -> dict[tuple[str, str], np.ndarray]:
    sums = {}
    for sex in ["m", "f"]:
        for group, ages in AGE_BINS.items():
            paths = [age_raster_path(iso, sex, age) for age in ages]
            existing = [p for p in paths if virtual_path_exists(p)]
            missing = [p for p in paths if not virtual_path_exists(p)]
            for p in missing:
                print(f"[WARN] Missing age-sex raster: {p}")
            if not existing:
                sums[(sex, group)] = np.full(n_cells, np.nan)
            else:
                print(f"{iso}: summing {len(existing)} rasters for {sex}/{group}")
                sums[(sex, group)] = zonal_sum_bincount(cell_id_flat, n_cells, existing)
    return sums


def validate_worldpop_country(out: pd.DataFrame) -> None:
    iso = str(out["country_iso3"].iloc[0])
    total = float(out["pop_count"].sum())
    max_pop = float(out["pop_count"].max())
    max_density = float(out["pop_density"].max())
    lo, hi = WORLDPOP_EXPECTED_TOTAL_RANGE.get(iso, (0, np.inf))

    print(f"{iso}: WorldPop total={total:,.0f}, max_cell_pop={max_pop:,.0f}, max_density={max_density:,.0f}/km2")
    if not (lo <= total <= hi):
        raise ValueError(
            f"{iso}: WorldPop total {total:,.0f} outside expected range [{lo:,.0f}, {hi:,.0f}]. "
            "Likely missing files or unmasked nodata/sentinel values."
        )
    if max_pop > 1_000_000 or max_density > 500_000:
        raise ValueError(
            f"{iso}: implausible max pop/density. Check nodata masking and raster alignment."
        )


def compute_worldpop_country(iso: str) -> pd.DataFrame:
    sub = cells[cells.country_iso3 == iso].copy().reset_index(drop=True)
    out = sub.drop(columns=["geometry"])[["country_iso3", "h3_index", "area_km2", "centroid_lat", "centroid_lon"]].copy()
    out["worldpop_effective_year"] = WORLDPOP_EFFECTIVE_YEAR[iso]

    ref = reference_raster_for_country(iso)
    print(f"{iso}: rasterizing {len(sub):,} H3 cells onto {ref}")
    cell_id, n_cells = make_cell_id_raster(sub, ref)

    print(f"{iso}: computing age-sex sums from {WORLDPOP_AGESEX_DIR.get(iso) or WORLDPOP_AGESEX_ZIP.get(iso)}")
    sums = age_sex_group_sums(iso, cell_id, n_cells)

    youth = sums[("m", "youth")] + sums[("f", "youth")]
    working = sums[("m", "working")] + sums[("f", "working")]
    elderly = sums[("m", "elderly")] + sums[("f", "elderly")]
    male = sums[("m", "youth")] + sums[("m", "working")] + sums[("m", "elderly")]
    female = sums[("f", "youth")] + sums[("f", "working")] + sums[("f", "elderly")]

    total_raster = WORLDPOP_TOTAL_RASTER.get(iso)
    if total_raster and virtual_path_exists(total_raster):
        print(f"{iso}: total population from standalone raster {total_raster}")
        pop = zonal_sum_bincount(cell_id, n_cells, [total_raster])
    else:
        print(f"{iso}: standalone total-pop raster unavailable; using summed age-sex rasters")
        pop = youth + working + elderly

    if np.all(~np.isfinite(pop)):
        raise FileNotFoundError(f"{iso}: no usable WorldPop total or age-sex rasters found")

    out["pop_count"] = np.nan_to_num(pop, nan=0.0).astype("float32")
    out["pop_density"] = (out["pop_count"] / out["area_km2"]).astype("float32")

    stable = np.isfinite(working) & (working >= DEPENDENCY_DENOM_MIN)  # avoid near-empty denominators
    with np.errstate(divide="ignore", invalid="ignore"):
        out["youth_dep_ratio"] = np.where(stable, np.minimum(youth / working, DEPENDENCY_CAP), np.nan)  # cap sparse-cell extremes
        out["elderly_dep_ratio"] = np.where(stable, np.minimum(elderly / working, DEPENDENCY_CAP), np.nan)
        out["sex_ratio"] = np.where(stable & (female > 0), male / female, np.nan)
    out["youth_dep_ratio"] = out["youth_dep_ratio"].astype("float32")
    out["elderly_dep_ratio"] = out["elderly_dep_ratio"].astype("float32")
    out["sex_ratio"] = out["sex_ratio"].astype("float32")

    validate_worldpop_country(out)
    return out


def add_market_access(df: pd.DataFrame) -> pd.DataFrame:
    """Distance-decayed population, with neighbor-only QA."""
    out_frames = []
    earth_radius_km = 6371.0088

    for iso, sub in df.groupby("country_iso3", sort=False):
        sub = sub.copy().reset_index(drop=True)
        coords_rad = np.deg2rad(sub[["centroid_lat", "centroid_lon"]].to_numpy(dtype="float64"))
        pop = sub["pop_count"].to_numpy(dtype="float64")
        area = sub["area_km2"].to_numpy(dtype="float64")
        self_dist_km = 0.5 * np.sqrt(area / np.pi)  # own mass at half equivalent radius
        tree = BallTree(coords_rad, metric="haversine")

        for d0 in MARKET_ACCESS_D0_KM:
            radius_km = d0 * MARKET_ACCESS_RADIUS_FACTOR  # stop after 5 decay lengths
            vals = np.zeros(len(sub), dtype="float64")
            vals_neighbor = np.zeros(len(sub), dtype="float64")
            print(f"{iso}: market access d0={d0}km, radius={radius_km}km, chunk={MARKET_ACCESS_QUERY_CHUNK:,}")

            for start in tqdm(range(0, len(sub), MARKET_ACCESS_QUERY_CHUNK), desc=f"{iso} market d0={d0}"):
                stop = min(start + MARKET_ACCESS_QUERY_CHUNK, len(sub))
                inds, dists = tree.query_radius(
                    coords_rad[start:stop],
                    r=radius_km / earth_radius_km,
                    return_distance=True,
                    sort_results=False,
                )

                for offset, (ii, dd) in enumerate(zip(inds, dists)):
                    i = start + offset
                    if len(ii) == 0:
                        continue
                    dist_km = dd * earth_radius_km
                    self_mask = ii == i
                    if self_mask.any():
                        dist_km = dist_km.copy()
                        dist_km[self_mask] = self_dist_km[i]
                    weights = np.exp(-dist_km / d0)
                    vals[i] = np.sum(pop[ii] * weights)
                    vals_neighbor[i] = np.sum(pop[ii[~self_mask]] * weights[~self_mask])

            sub[f"market_access_pop_exp_{d0}km"] = vals.astype("float32")
            sub[f"market_access_neighbor_pop_exp_{d0}km"] = vals_neighbor.astype("float32")

        out_frames.append(sub)

    return pd.concat(out_frames, ignore_index=True)


if WORLDPOP_OUT.exists():
    worldpop = pd.read_parquet(WORLDPOP_OUT)
    print(f"Loaded WorldPop features: {len(worldpop):,}")
else:
    frames = []
    for iso in cells.country_iso3.unique():
        frames.append(compute_worldpop_country(iso))
    worldpop = pd.concat(frames, ignore_index=True)
    worldpop = add_market_access(worldpop)
    keep = [
        "country_iso3", "h3_index", "worldpop_effective_year",
        "pop_count", "pop_density", "youth_dep_ratio", "elderly_dep_ratio", "sex_ratio",
        "market_access_pop_exp_10km", "market_access_pop_exp_20km", "market_access_pop_exp_30km",
        "market_access_neighbor_pop_exp_10km", "market_access_neighbor_pop_exp_20km", "market_access_neighbor_pop_exp_30km",
    ]
    worldpop = worldpop[keep]
    worldpop.to_parquet(WORLDPOP_OUT, compression="snappy", index=False)
    print(f"Wrote WorldPop features: {len(worldpop):,} -> {WORLDPOP_OUT}")

display(worldpop.groupby("country_iso3")[["pop_count", "pop_density"]].agg(["sum", "mean", "median", "max"]).round(2))
top_cols = ["country_iso3", "h3_index", "pop_count", "pop_density", "market_access_pop_exp_30km"]
display(worldpop.sort_values("pop_count", ascending=False)[top_cols].head(10))
worldpop.head()

## OSM roads

v1 notes:

- segment midpoint sets the H3 cell, so long crossings can be missed
- node degree uses rounded endpoints; unsplit crossings stay unresolved
- `road_grid_like_index` is QA only, raw topology fields stay primary

In [ ]:
ROAD_CLASS_RANK = {
    "motorway": 6, "trunk": 6,
    "primary": 5,
    "secondary": 4,
    "tertiary": 3,
    "residential": 2, "unclassified": 2, "living_street": 2,
    "service": 1, "track": 1, "path": 1, "footway": 1, "cycleway": 1,
}
INTERSECTION_COORD_PRECISION = 7


def haversine_m(lon1, lat1, lon2, lat2):
    r = 6371008.8
    lon1 = np.deg2rad(lon1); lat1 = np.deg2rad(lat1)
    lon2 = np.deg2rad(lon2); lat2 = np.deg2rad(lat2)
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


def segment_bearing_deg(lon1, lat1, lon2, lat2):
    lon1 = np.deg2rad(lon1); lat1 = np.deg2rad(lat1)
    lon2 = np.deg2rad(lon2); lat2 = np.deg2rad(lat2)
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    bearing = (np.rad2deg(np.arctan2(x, y)) + 360) % 180
    return bearing


def orientation_entropy_deg(angles, weights=None, bins=18):
    angles = np.asarray(angles, dtype="float64")
    mask = np.isfinite(angles)
    angles = angles[mask]
    if weights is not None:
        weights = np.asarray(weights, dtype="float64")[mask]
    if len(angles) < 3:
        return np.nan
    hist, _ = np.histogram(angles, bins=bins, range=(0, 180), weights=weights)
    if hist.sum() <= 0:
        return np.nan
    p = hist[hist > 0] / hist.sum()
    return float(-(p * np.log(p)).sum() / np.log(bins))


def load_osm_roads(iso: str) -> gpd.GeoDataFrame:
    gpkg = OSM_GPKG[iso]
    if not virtual_path_exists(gpkg):
        raise FileNotFoundError(f"Missing OSM GPKG for {iso}: {gpkg}")
    print(f"{iso}: reading roads from {gpkg}")
    roads = pyogrio.read_dataframe(str(gpkg), layer="gis_osm_roads_free", columns=["fclass"])
    roads = roads[roads.geometry.notna() & ~roads.geometry.is_empty].copy()
    roads["rank"] = roads["fclass"].map(ROAD_CLASS_RANK).fillna(0).astype("int8")
    roads = roads[roads["rank"] > 0].copy()
    print(f"{iso}: {len(roads):,} roads after rank filter")
    return roads


def compute_osm_roads_country(iso: str) -> pd.DataFrame:
    subcells = cells[cells.country_iso3 == iso].drop(columns=["geometry"])[[
        "country_iso3", "h3_index", "area_km2"
    ]].copy()
    roads = load_osm_roads(iso)

    coords, road_idx = shapely.get_coordinates(roads.geometry.values, return_index=True)
    same_road = road_idx[:-1] == road_idx[1:]
    seg_start = coords[:-1][same_road]
    seg_end = coords[1:][same_road]
    seg_road = road_idx[:-1][same_road]
    seg_rank = roads["rank"].to_numpy()[seg_road]
    seg_len_m = haversine_m(seg_start[:, 0], seg_start[:, 1], seg_end[:, 0], seg_end[:, 1])

    keep = np.isfinite(seg_len_m) & (seg_len_m > 0)
    seg_start = seg_start[keep]
    seg_end = seg_end[keep]
    seg_rank = seg_rank[keep]
    seg_len_m = seg_len_m[keep]

    # midpoint assigns each segment once; long crossings can be missed
    mid_lon = (seg_start[:, 0] + seg_end[:, 0]) / 2
    mid_lat = (seg_start[:, 1] + seg_end[:, 1]) / 2
    seg_h3 = [h3.latlng_to_cell(float(lat), float(lon), H3_RES) for lat, lon in tqdm(zip(mid_lat, mid_lon), total=len(mid_lat), desc=f"{iso} segment h3")]
    bearing = segment_bearing_deg(seg_start[:, 0], seg_start[:, 1], seg_end[:, 0], seg_end[:, 1])

    seg_df = pd.DataFrame({
        "h3_index": seg_h3,
        "road_total_length_m": seg_len_m,
        "road_class_max": seg_rank,
        "bearing": bearing,
    })

    road_agg = (
        seg_df.groupby("h3_index", sort=False)
        .agg(
            road_total_length_m=("road_total_length_m", "sum"),
            road_class_max=("road_class_max", "max"),
            road_segment_count=("road_total_length_m", "size"),
            road_bearing_entropy=("bearing", lambda s: orientation_entropy_deg(s.to_numpy())),
        )
        .reset_index()
    )

    endpoints = np.round(np.concatenate([seg_start, seg_end]), INTERSECTION_COORD_PRECISION)  # merge near-identical endpoints only
    unique_pts, degree = np.unique(endpoints, axis=0, return_counts=True)
    node_h3 = [h3.latlng_to_cell(float(lat), float(lon), H3_RES) for lon, lat in tqdm(unique_pts, desc=f"{iso} node h3")]
    node_df = pd.DataFrame({"h3_index": node_h3, "degree": degree})
    node_df["degree_5plus"] = node_df["degree"] >= 5

    node_agg = (
        node_df.groupby("h3_index", sort=False)
        .agg(
            road_node_count=("degree", "size"),
            road_dead_end_count=("degree", lambda s: int((s == 1).sum())),
            road_degree2_count=("degree", lambda s: int((s == 2).sum())),
            road_degree3_count=("degree", lambda s: int((s == 3).sum())),
            road_degree4_count=("degree", lambda s: int((s == 4).sum())),
            road_degree5plus_count=("degree_5plus", "sum"),
            road_intersection_count=("degree", lambda s: int((s >= 3).sum())),
            road_degree_mean=("degree", "mean"),
        )
        .reset_index()
    )

    out = subcells.merge(road_agg, on="h3_index", how="left").merge(node_agg, on="h3_index", how="left")
    count_cols = [
        "road_total_length_m", "road_segment_count", "road_node_count", "road_dead_end_count",
        "road_degree2_count", "road_degree3_count", "road_degree4_count", "road_degree5plus_count",
        "road_intersection_count",
    ]
    for c in count_cols:
        out[c] = out[c].fillna(0)
    out["road_class_max"] = out["road_class_max"].fillna(0).astype("int8")
    out["road_density_km_per_km2"] = (out["road_total_length_m"] / 1000.0 / out["area_km2"]).astype("float32")
    out["road_dead_end_ratio"] = np.where(out["road_node_count"] > 0, out["road_dead_end_count"] / out["road_node_count"], np.nan).astype("float32")
    out["road_degree3_share"] = np.where(out["road_node_count"] > 0, out["road_degree3_count"] / out["road_node_count"], np.nan).astype("float32")
    out["road_degree4_share"] = np.where(out["road_node_count"] > 0, out["road_degree4_count"] / out["road_node_count"], np.nan).astype("float32")
    out["road_degree5plus_share"] = np.where(out["road_node_count"] > 0, out["road_degree5plus_count"] / out["road_node_count"], np.nan).astype("float32")
    out["road_beta_connectivity"] = np.where(out["road_node_count"] > 0, out["road_segment_count"] / out["road_node_count"], np.nan).astype("float32")
    # QA composite only
    out["road_grid_like_index"] = (out["road_degree4_share"] - out["road_dead_end_ratio"]).astype("float32")
    out["osm_effective_source"] = str(OSM_GPKG[iso])

    keep = [
        "country_iso3", "h3_index",
        "road_density_km_per_km2", "road_class_max", "road_segment_count",
        "road_node_count", "road_intersection_count", "road_dead_end_count",
        "road_dead_end_ratio", "road_degree_mean", "road_degree3_share",
        "road_degree4_share", "road_degree5plus_share", "road_beta_connectivity",
        "road_grid_like_index", "road_bearing_entropy", "osm_effective_source",
    ]
    return out[keep]


if OSM_OUT.exists():
    osm = pd.read_parquet(OSM_OUT)
    print(f"Loaded OSM features: {len(osm):,}")
else:
    frames = []
    for iso in cells.country_iso3.unique():
        try:
            frames.append(compute_osm_roads_country(iso))
        except FileNotFoundError as e:
            print(f"[WARN] {e}")
    osm = pd.concat(frames, ignore_index=True)
    osm.to_parquet(OSM_OUT, compression="snappy", index=False)
    print(f"Wrote OSM features: {len(osm):,} -> {OSM_OUT}")

## Local feature QA

Coverage, missingness and extreme-value checks after each local table.

In [ ]:
def show_if_exists(label: str, path: Path | None) -> pd.DataFrame | None:
    exists = bool(path is not None and Path(path).exists())
    print(f"{label:8s}: {path} exists={exists}")
    if not exists:
        return None
    df = pd.read_parquet(path)
    print(f"  shape={df.shape}")
    display(df.head())
    return df


rwi_check = show_if_exists("RWI", RWI_OUT)
dhs_check = show_if_exists("DHS", DHS_OUT)
wp_check = show_if_exists("WorldPop", WORLDPOP_OUT)
osm_check = show_if_exists("OSM", OSM_OUT)

if rwi_check is not None:
    display(rwi_check.groupby("country_iso3").agg(
        n_cells=("h3_index", "size"),
        cells_with_rwi=("has_rwi", "sum"),
        rwi_points=("rwi_n_points", "sum"),
        rwi_mean=("rwi_mean", "mean"),
    ).round(3))

if dhs_check is not None:
    display(dhs_check.groupby("country_iso3").agg(
        n_cells=("h3_index", "size"),
        cells_with_dhs=("has_dhs", "sum"),
        dhs_clusters=("dhs_cluster_count", "sum"),
    ))

if wp_check is not None:
    display(wp_check.groupby("country_iso3").agg(
        pop_total=("pop_count", "sum"),
        pop_max=("pop_count", "max"),
        pop_density_max=("pop_density", "max"),
        ma30_p50=("market_access_pop_exp_30km", "median"),
        ma30_max=("market_access_pop_exp_30km", "max"),
    ).round(2))
    display(wp_check.sort_values("pop_count", ascending=False).head(20)[
        ["country_iso3", "h3_index", "pop_count", "pop_density", "market_access_pop_exp_30km"]
    ])

if osm_check is not None:
    display(osm_check.groupby("country_iso3").agg(
        n_cells=("h3_index", "size"),
        cells_with_roads=("road_segment_count", lambda s: int((s > 0).sum())),
        road_density_p50=("road_density_km_per_km2", "median"),
        road_density_max=("road_density_km_per_km2", "max"),
        dead_end_p50=("road_dead_end_ratio", "median"),
        degree4_p50=("road_degree4_share", "median"),
    ).round(3))


## NGA/TZA GOB morphology

Build the same H3 morphology fields from the existing NGA/TZA CSVs, then append to the active-country GOB table.

In [ ]:
# NGA/TZA GOB morphology

GOB_CONFIDENCE_MIN = 0.70
GOB_CHUNK_ROWS = 1_000_000
GOB_ORIENTATION_BINS = 18
COMPUTE_GOB_ORIENTATION = True
GOB_LOCAL_COUNTRIES = ["NGA", "TZA"]

GOB_LOCAL_CSV = {
    "NGA": _first_existing(
        PROJECT_ROOT / "open_buildings_v3_polygons_ne_10m_NGA.csv",
        PROJECT_ROOT / "gob" / "open_buildings_v3_polygons_ne_10m_NGA.csv",
        PROJECT_ROOT / "open_buildings_v3_polygons_ne_10m_NGA.csv.gz",
    ),
    "TZA": _first_existing(
        PROJECT_ROOT / "open_buildings_v3_polygons_ne_10m_TZA.csv",
        PROJECT_ROOT / "gob" / "open_buildings_v3_polygons_ne_10m_TZA.csv",
        PROJECT_ROOT / "open_buildings_v3_polygons_ne_10m_TZA.csv.gz",
    ),
}


def gob_gini(values) -> float:
    values = np.asarray(values, dtype="float64")
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    if np.any(values < 0):
        values = values - np.nanmin(values)
    total = values.sum()
    if total <= 0:
        return np.nan
    values = np.sort(values)
    n = values.size
    return float((2 * np.arange(1, n + 1) @ values) / (n * total) - (n + 1) / n)


def gob_normalized_entropy(angles, bins: int = GOB_ORIENTATION_BINS) -> float:
    angles = np.asarray(angles, dtype="float64")
    angles = angles[np.isfinite(angles)]
    if angles.size < 3:
        return np.nan
    hist, _ = np.histogram(angles % 180.0, bins=bins, range=(0, 180))
    p = hist[hist > 0] / hist.sum()
    return float(-(p * np.log(p)).sum() / np.log(bins))


def gob_building_orientations_deg(wkt_values) -> np.ndarray:
    """Longest MRR edge angle, vectorized."""
    geoms = shapely.from_wkt(np.asarray(wkt_values, dtype=object), on_invalid="ignore")
    rect = shapely.oriented_envelope(geoms)
    coords, idx = shapely.get_coordinates(rect, return_index=True)
    out = np.full(len(geoms), np.nan)
    if coords.shape[0] < 2:
        return out
    same = idx[:-1] == idx[1:]
    d = coords[1:][same] - coords[:-1][same]
    gi = idx[:-1][same]
    seg_len = np.hypot(d[:, 0], d[:, 1])
    seg_ang = np.degrees(np.arctan2(d[:, 1], d[:, 0])) % 180.0
    order = np.lexsort((seg_len, gi))
    gis, angs = gi[order], seg_ang[order]
    last = np.empty(gis.shape[0], dtype=bool)
    last[-1] = True
    last[:-1] = gis[1:] != gis[:-1]
    out[gis[last]] = angs[last]
    return out


def compute_local_gob_morphology_for_country(iso: str) -> pd.DataFrame:
    path = GOB_LOCAL_CSV.get(iso)
    if path is None or not Path(path).exists():
        raise FileNotFoundError(f"Missing local GOB CSV for {iso}: {path}")

    usecols = ["latitude", "longitude", "area_in_meters", "confidence"]
    if COMPUTE_GOB_ORIENTATION:
        usecols.append("geometry")
    dtypes = {
        "latitude": "float64",
        "longitude": "float64",
        "area_in_meters": "float64",
        "confidence": "float64",
    }

    records = []
    print(f"{iso}: reading local GOB {path}")
    for chunk in pd.read_csv(path, chunksize=GOB_CHUNK_ROWS, compression="infer", usecols=usecols, dtype=dtypes):
        chunk = chunk[chunk["confidence"] >= GOB_CONFIDENCE_MIN]
        if chunk.empty:
            continue

        lat = chunk["latitude"].to_numpy(dtype="float64")
        lon = chunk["longitude"].to_numpy(dtype="float64")
        h3idx = [h3.latlng_to_cell(float(a), float(o), H3_RES) for a, o in zip(lat, lon)]
        orient = (
            gob_building_orientations_deg(chunk["geometry"].to_numpy())
            if COMPUTE_GOB_ORIENTATION else np.full(len(chunk), np.nan)
        )
        records.append(pd.DataFrame({
            "h3_index": h3idx,
            "area_in_meters": chunk["area_in_meters"].to_numpy(dtype="float64"),
            "centroid_lat": lat,
            "centroid_lon": lon,
            "_orientation": orient,
        }))

    if not records:
        return pd.DataFrame(columns=["country_iso3", "h3_index"])

    b = pd.concat(records, ignore_index=True)
    rows = []
    for h, g in tqdm(b.groupby("h3_index", sort=False), desc=f"{iso} GOB cells"):
        n = len(g)
        area = g["area_in_meters"].to_numpy(dtype="float64")

        nn_cv = np.nan
        if n >= 3:
            lat = np.deg2rad(g["centroid_lat"].to_numpy(dtype="float64"))
            lon = np.deg2rad(g["centroid_lon"].to_numpy(dtype="float64"))
            x = 6371000.0 * lon * np.cos(np.nanmean(lat))
            y = 6371000.0 * lat
            pts = np.column_stack([x, y])
            dists, _ = cKDTree(pts).query(pts, k=2)
            nn = dists[:, 1]
            mean_nn = np.nanmean(nn)
            if mean_nn > 0:
                nn_cv = float(np.nanstd(nn) / mean_nn)

        rows.append({
            "country_iso3": iso,
            "h3_index": h,
            "gob_building_count": int(n),
            "gob_building_area_total_m2": float(np.nansum(area)),
            "gob_building_area_mean_m2": float(np.nanmean(area)) if n else np.nan,
            "gob_building_area_std_m2": float(np.nanstd(area)) if n else np.nan,
            "gob_building_size_gini": gob_gini(area),
            "gob_nn_cv": nn_cv,
            "gob_spacing_regularity": float(1.0 / (1.0 + nn_cv)) if np.isfinite(nn_cv) else np.nan,
            "gob_orientation_entropy": gob_normalized_entropy(g["_orientation"].to_numpy(dtype="float64")),
            "gob_effective_year": 2023,
        })

    return pd.DataFrame(rows)


def finalize_gob_country_frame(raw_country: pd.DataFrame, iso: str) -> pd.DataFrame:
    spine = cells[cells["country_iso3"] == iso].drop(columns=["geometry"])[["country_iso3", "h3_index", "area_km2"]]
    out = spine.merge(raw_country, on=["country_iso3", "h3_index"], how="left", validate="one_to_one")
    out["gob_building_count"] = out["gob_building_count"].fillna(0).astype("int32")
    out["gob_building_area_total_m2"] = out["gob_building_area_total_m2"].fillna(0).astype("float32")
    out["gob_building_density_per_km2"] = (out["gob_building_count"] / out["area_km2"]).astype("float32")
    out["gob_footprint_fraction"] = (
        out["gob_building_area_total_m2"] / (out["area_km2"] * 1_000_000.0)
    ).clip(0, 1).astype("float32")
    out["gob_effective_year"] = out["gob_effective_year"].fillna(2023).astype("int16")
    return out.drop(columns=["area_km2"])


def update_gob_morphology_with_local_nga_tza(overwrite_existing_local: bool = False) -> pd.DataFrame:
    active_countries = sorted(cells["country_iso3"].unique())
    print(f"Active countries from local spine: {active_countries}")

    if GOB_MORPH_PATH.exists():
        base = pd.read_parquet(GOB_MORPH_PATH)
        print(f"Loaded existing GOB morphology: {len(base):,} rows -> {GOB_MORPH_PATH}")
    else:
        base = pd.DataFrame(columns=["country_iso3", "h3_index"])
        print(f"No existing GOB morphology found; creating {GOB_MORPH_PATH}")

    base = base[base["country_iso3"].isin(active_countries)].copy()
    base_countries = set(base["country_iso3"].dropna().unique())
    need = [iso for iso in GOB_LOCAL_COUNTRIES if overwrite_existing_local or iso not in base_countries]
    print(f"Local GOB countries to compute: {need}")

    local_frames = []
    for iso in need:
        raw_country = compute_local_gob_morphology_for_country(iso)
        local_frames.append(finalize_gob_country_frame(raw_country, iso))

    if local_frames:
        base = base[~base["country_iso3"].isin([f["country_iso3"].iloc[0] for f in local_frames])]
        combined = pd.concat([base] + local_frames, ignore_index=True, sort=False)
    else:
        combined = base

    combined = combined[combined["country_iso3"].isin(active_countries)].copy()
    dup = combined.duplicated(["country_iso3", "h3_index"]).sum()
    if dup:
        raise ValueError(f"GOB morphology has duplicate keys after update: {dup:,}")

    coverage = combined.groupby("country_iso3").size().rename("n_gob_rows")
    expected = cells.groupby("country_iso3").size().rename("n_spine_rows")
    check = pd.concat([expected, coverage], axis=1).fillna(0).astype(int)
    check["matches_spine"] = check["n_gob_rows"] == check["n_spine_rows"]
    display(check)

    missing = sorted(set(active_countries) - set(combined["country_iso3"].unique()))
    if missing:
        raise ValueError(f"GOB morphology still missing active countries: {missing}")
    if not check["matches_spine"].all():
        raise ValueError("GOB morphology row counts do not match the active H3 spine.")

    combined.to_parquet(GOB_MORPH_PATH, compression="snappy", index=False)
    print(f"Wrote updated GOB morphology: {len(combined):,} rows -> {GOB_MORPH_PATH}")
    return combined


# keep existing NGA/TZA rows
gob_morph = update_gob_morphology_with_local_nga_tza(overwrite_existing_local=False)

## Final merge

GEE spine plus `local_outputs/*.parquet`. GOB joins when present. output is `cells_full_tabular_v2.parquet`.

In [ ]:
def optional_read_parquet(path: Path | None, label: str) -> pd.DataFrame | None:
    exists = bool(path is not None and Path(path).exists())
    if exists:
        df = pd.read_parquet(path)
        print(f"Loaded {label}: {len(df):,} rows, {len(df.columns):,} columns")
        return df
    print(f"[WARN] Missing {label}: {path}")
    return None


if not GEE_FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"Missing GEE features: {GEE_FEATURES_PATH}"
    )

gee = pd.read_parquet(GEE_FEATURES_PATH)
print(f"GEE features: {len(gee):,} rows, {len(gee.columns):,} columns, "
      f"countries={sorted(gee['country_iso3'].unique())}")

# active H3 spine stays the row anchor
final = cells.drop(columns=["geometry"])[["country_iso3", "h3_index"]].copy()
final = final.merge(gee, on=["country_iso3", "h3_index"], how="left", validate="one_to_one")
_missing_gee = sorted(set(final["country_iso3"].unique()) - set(gee["country_iso3"].unique()))
print(f"Base spine + GEE: {len(final):,} rows; countries without GEE features: {_missing_gee}")

for path, label in [
    (GOB_MORPH_PATH, "GOB morphology"),
    (RWI_OUT, "RWI local features"),
    (DHS_OUT, "DHS local features"),
    (WORLDPOP_OUT, "WorldPop local features"),
    (OSM_OUT, "OSM local features"),
]:
    df = optional_read_parquet(path, label)
    if df is None:
        continue
    before = len(final)
    final = final.merge(df, on=["country_iso3", "h3_index"], how="left", validate="one_to_one")
    assert len(final) == before, f"{label} changed row count"

for c in [c for c in final.columns if c.startswith("ae_A")]:
    final[c] = pd.to_numeric(final[c], errors="coerce").astype("float32")

expected_final_countries = sorted(cells["country_iso3"].unique())
actual_final_countries = sorted(final["country_iso3"].unique())
if actual_final_countries != expected_final_countries:
    raise ValueError(f"Final country mismatch. Expected {expected_final_countries}, got {actual_final_countries}")
if len(final) != len(cells):
    raise ValueError(f"Final row count mismatch. Expected {len(cells):,}, got {len(final):,}")

FINAL_PATH.parent.mkdir(parents=True, exist_ok=True)
final.to_parquet(FINAL_PATH, compression="snappy", index=False)

print(f"Expected active countries: {expected_final_countries}")
print(f"Final table: {len(final):,} rows, {len(final.columns):,} columns")